In [0]:
%sql
MERGE INTO value_stream_monitoring.databricks_cost_fact t
USING (
    SELECT Date, Meter, SUM(Cost) AS Cost, 'License' AS pricing_type
    FROM value_stream_monitoring.databricks_costs_daily
    WHERE Date >= date_sub(current_date(), 7)
    GROUP BY Date, Meter
    UNION ALL
    SELECT Date, Meter, SUM(Cost) AS Cost, 'Infra' AS pricing_type
    FROM value_stream_monitoring.databricks_infra_costs_daily
    WHERE Date >= date_sub(current_date(), 7)
    GROUP BY Date, Meter
    UNION ALL
    SELECT Date, Meter, SUM(Cost) AS Cost, 'Storage' AS pricing_type
    FROM value_stream_monitoring.databricks_storage_costs_daily
    WHERE Date >= date_sub(current_date(), 7)
    GROUP BY Date, Meter
) s
ON t.Date >= date_sub(current_date(), 7)
AND t.Date = s.Date
AND t.Meter = s.Meter
AND t.pricing_type = s.pricing_type
WHEN MATCHED AND t.Cost <> s.Cost THEN
    UPDATE SET t.Cost = s.Cost
WHEN NOT MATCHED THEN
    INSERT (Date, Meter, Cost, pricing_type)
    VALUES (s.Date, s.Meter, s.Cost, s.pricing_type)
WHEN NOT MATCHED BY SOURCE AND t.Date >= date_sub(current_date(), 7) THEN
    DELETE